# NewsPulse — Analisis Tren Berita Nasional
**Topik 5 — ETS Big Data Kelompok 4**

1. Kata paling sering muncul di judul berita (Top 15)
2. Distribusi berita per sumber
3. Volume publikasi per jam
4. **BONUS** — K-Means Clustering berbasis TF-IDF judul

In [15]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, explode, split, lower, trim, count, hour,
    to_timestamp, desc, lit, regexp_replace, length, coalesce
)
import json, os

try: SparkSession.builder.getOrCreate().stop()
except: pass

spark = SparkSession.builder \
    .appName("NewsPulse Analysis") \
    .master("local[*]") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true") \
    .getOrCreate()

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.sparkContext.setLogLevel("WARN")
print("SparkSession aktif:", spark.version)
print("timeParserPolicy:", spark.conf.get("spark.sql.legacy.timeParserPolicy"))
print("ansi.enabled:", spark.conf.get("spark.sql.ansi.enabled"))


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
SparkSession aktif: 4.1.1
timeParserPolicy: LEGACY
ansi.enabled: false


In [16]:
# Membaca data dari HDFS
HDFS_API = "hdfs://localhost:8020/data/news/api/"
HDFS_RSS = "hdfs://localhost:8020/data/news/rss/"

df_api = spark.read.option("multiLine", True).json(HDFS_API)
print(f"Data API dari HDFS: {df_api.count()} records")

df_rss = spark.read.option("multiLine", True).json(HDFS_RSS)
print(f"Data RSS dari HDFS: {df_rss.count()} records")

common_cols = ["judul","sumber","url","kategori","deskripsi","waktu_terbit","timestamp"]
for c in common_cols:
    if c not in df_api.columns: df_api = df_api.withColumn(c, lit(""))
    if c not in df_rss.columns: df_rss = df_rss.withColumn(c, lit(""))

df_all = df_api.select(common_cols).union(df_rss.select(common_cols)).dropDuplicates(["url"])
print(f"Total gabungan: {df_all.count()}")
df_all.show(5, truncate=50)
df_all.createOrReplaceTempView("news_view")

Data API dari HDFS: 46 records
Data RSS dari HDFS: 155 records
Total gabungan: 82
+--------------------------------------------------+--------------------+--------------------------------------------------+--------+--------------------------------------------------+--------------------+--------------------------+
|                                             judul|              sumber|                                               url|kategori|                                         deskripsi|        waktu_terbit|                 timestamp|
+--------------------------------------------------+--------------------+--------------------------------------------------+--------+--------------------------------------------------+--------------------+--------------------------+
|Soal Kelahiran Anak Al dan Alyssa, Ahmad Dhani:...|          Kompas.com|https://entertainment.kompas.com/read/2026/05/0...|nasional|Ahmad Dhani tak sabar menanti cucu pertama dari...|2026-05-03T14:55:00Z|2026-05-04T10:

---
## Analisis 1 — Kata Paling Sering Muncul di Judul (Top 15)

In [17]:
STOPWORDS = ["dan","yang","di","ke","dari","untuk","dengan","ini","itu","ada",
    "akan","juga","tidak","se","pada","oleh","secara","sebuah","tersebut",
    "telah","saat","bisa","lebih","masih","sudah","hari","kota","dalam",
    "tahun","atau","ia","kami","kita","mereka","dia","atas","hal",
    "the","of","in","a","to","is","an","-","|","","soal","usai"]

df_kata = (
    df_all
    .select(explode(split(lower(col("judul")), r"\s+")).alias("kata"))
    .withColumn("kata", trim(regexp_replace(col("kata"), r"[^a-zA-Z0-9]", "")))
    .filter(~col("kata").isin(STOPWORDS)).filter(length(col("kata")) > 1)
    .groupBy("kata").agg(count("*").alias("frekuensi"))
    .orderBy(desc("frekuensi")).limit(15)
)
print("Top 15 Kata Trending — DataFrame API:")
df_kata.show(15, truncate=False)

Top 15 Kata Trending — DataFrame API:
+----------+---------+
|kata      |frekuensi|
+----------+---------+
|2026      |11       |
|jadi      |7        |
|megawati  |7        |
|prabowo   |7        |
|hardiknas |6        |
|amien     |5        |
|lpdp      |5        |
|pembekalan|5        |
|rais      |5        |
|demo      |5        |
|anak      |4        |
|tni       |4        |
|baru      |4        |
|partai    |4        |
|cara      |4        |
+----------+---------+



In [18]:
spark.sql("""
SELECT kata, COUNT(*) AS frekuensi FROM (
  SELECT TRIM(REGEXP_REPLACE(word, '[^a-zA-Z0-9]', '')) AS kata FROM (
    SELECT EXPLODE(SPLIT(LOWER(judul), '\\s+')) AS word FROM news_view
  )
) WHERE LENGTH(kata) > 1 AND kata NOT IN (
  'dan','yang','di','ke','dari','untuk','dengan','ini','itu','ada',
  'akan','juga','tidak','se','pada','oleh','secara','sebuah','tersebut',
  'telah','saat','bisa','lebih','masih','sudah','hari','kota','dalam',
  'tahun','atau','ia','soal','usai')
GROUP BY kata ORDER BY frekuensi DESC LIMIT 15
""").show(15, truncate=False)

+------------------------+---------+
|kata                    |frekuensi|
+------------------------+---------+
|pe                      |3        |
|auru                    |2        |
|2026                    |2        |
|ipemerintahdaerahberpre |2        |
|re                      |2        |
|pon                     |2        |
|alebihpanjangkokbi      |2        |
|abem                    |2        |
|ilmuwanmenyebuttanpadino|2        |
|iabi                    |2        |
|umurmanu                |2        |
|demohardikna            |2        |
|alurkanbantuanuntukma   |2        |
|ma                      |2        |
|ta                      |2        |
+------------------------+---------+



### Interpretasi Bisnis — Analisis 1
Kata-kata trending mencerminkan isu utama media nasional:
1. **Identifikasi isu viral** — frekuensi tertinggi = topik dengan coverage terbanyak
2. **Press release** — sisipkan kata trending agar lebih newsworthy
3. **Monitoring reputasi** — deteksi jika brand muncul di top list
4. **Content planning** — buat konten yang align dengan tren

---
## Analisis 2 — Distribusi Berita per Sumber

In [19]:
df_sumber = df_all.groupBy("sumber").agg(count("*").alias("jumlah")).orderBy(desc("jumlah"))
print("Distribusi Berita per Sumber — DataFrame API:")
df_sumber.show(truncate=False)

Distribusi Berita per Sumber — DataFrame API:
+------------------------+------+
|sumber                  |jumlah|
+------------------------+------+
|Tempo                   |55    |
|Kompas.com              |5     |
|detikNews               |4     |
|detiksport              |3     |
|Qoo Media               |2     |
|CNBC Indonesia          |2     |
|RRI.co.id               |2     |
|detikHOT                |2     |
|Good News From Indonesia|1     |
|USAID IUWASH Tangguh    |1     |
|PWMU.CO                 |1     |
|tandaseru.id            |1     |
|ANTARA News             |1     |
|Goal.com                |1     |
|Kompas.tv               |1     |
+------------------------+------+



In [20]:
spark.sql("SELECT sumber, COUNT(*) AS jumlah FROM news_view GROUP BY sumber ORDER BY jumlah DESC").show(truncate=False)

+------------------------+------+
|sumber                  |jumlah|
+------------------------+------+
|Tempo                   |55    |
|Kompas.com              |5     |
|detikNews               |4     |
|detiksport              |3     |
|Qoo Media               |2     |
|CNBC Indonesia          |2     |
|RRI.co.id               |2     |
|detikHOT                |2     |
|Good News From Indonesia|1     |
|USAID IUWASH Tangguh    |1     |
|PWMU.CO                 |1     |
|tandaseru.id            |1     |
|ANTARA News             |1     |
|Goal.com                |1     |
|Kompas.tv               |1     |
+------------------------+------+



### Interpretasi Bisnis — Analisis 2
1. **Media placement** — sumber volume tinggi cocok untuk press release
2. **Diversifikasi** — jika satu sumber dominan, perluas jaringan media
3. **Partner strategis** — media produktif cocok untuk exclusive story
4. **Competitive intelligence** — bandingkan editorial preference antar media

---
## Analisis 3 — Volume Publikasi per Jam

In [21]:
df_jam = (
    df_all
    .withColumn("ts_parsed", coalesce(
        to_timestamp(col("waktu_terbit")),
        to_timestamp(col("waktu_terbit"), "EEE, d MMM yyyy HH:mm:ss Z"),
        to_timestamp(col("timestamp"))
    ))
    .withColumn("jam", hour(col("ts_parsed")))
    .filter(col("jam").isNotNull())
    .groupBy("jam").agg(count("*").alias("jumlah_berita"))
    .orderBy("jam")
)
print("Volume Publikasi per Jam — DataFrame API:")
df_jam.show(24, truncate=False)

Volume Publikasi per Jam — DataFrame API:
+---+-------------+
|jam|jumlah_berita|
+---+-------------+
|6  |5            |
|7  |2            |
|8  |4            |
|9  |1            |
|10 |4            |
|11 |4            |
|12 |5            |
|13 |4            |
|14 |3            |
|15 |2            |
|16 |4            |
|17 |4            |
|18 |2            |
|19 |5            |
|20 |5            |
|21 |14           |
|22 |8            |
|23 |6            |
+---+-------------+



In [22]:
spark.sql("""
SELECT HOUR(COALESCE(
  TO_TIMESTAMP(waktu_terbit),
  TO_TIMESTAMP(waktu_terbit, 'EEE, d MMM yyyy HH:mm:ss Z'),
  TO_TIMESTAMP(timestamp)
)) AS jam, COUNT(*) AS jumlah_berita
FROM news_view GROUP BY jam HAVING jam IS NOT NULL ORDER BY jam
""").show(24, truncate=False)

+---+-------------+
|jam|jumlah_berita|
+---+-------------+
|6  |5            |
|7  |2            |
|8  |4            |
|9  |1            |
|10 |4            |
|11 |4            |
|12 |5            |
|13 |4            |
|14 |3            |
|15 |2            |
|16 |4            |
|17 |4            |
|18 |2            |
|19 |5            |
|20 |5            |
|21 |14           |
|22 |8            |
|23 |6            |
+---+-------------+



### Interpretasi Bisnis — Analisis 3
1. **Timing press release** — kirim 1-2 jam sebelum peak hour
2. **Social media scheduling** — posting pada jam volume tinggi
3. **Monitoring shift** — fokus resource pada jam sibuk
4. **Breaking news window** — jam sepi cocok untuk cerita eksklusif

---
## Simpan Hasil ke HDFS & Dashboard

In [23]:
# Simpan ke HDFS via docker exec (reliable dari host Windows)
import subprocess, tempfile

def save_df_to_hdfs(df, hdfs_path, name):
    """Simpan DataFrame ke HDFS via docker exec"""
    rows = df.toPandas().to_dict(orient="records")
    tmp = os.path.join(tempfile.gettempdir(), f"{name}.json")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False)
    # Copy ke container
    subprocess.run(["docker", "cp", tmp, f"hadoop-namenode:/tmp/{name}.json"], check=True)
    # Mkdir + put di HDFS
    subprocess.run(["docker", "exec", "hadoop-namenode", "hdfs", "dfs", "-mkdir", "-p", hdfs_path], check=True)
    subprocess.run(["docker", "exec", "hadoop-namenode", "hdfs", "dfs", "-put", "-f",
                    f"/tmp/{name}.json", f"{hdfs_path}/{name}.json"], check=True)
    print(f"  [HDFS] {hdfs_path}/{name}.json tersimpan")

save_df_to_hdfs(df_kata, "/data/news/hasil", "kata_trending")
save_df_to_hdfs(df_sumber, "/data/news/hasil", "distribusi_sumber")
save_df_to_hdfs(df_jam, "/data/news/hasil", "volume_per_jam")
print("Semua hasil analisis tersimpan ke HDFS!")

  [HDFS] /data/news/hasil/kata_trending.json tersimpan
  [HDFS] /data/news/hasil/distribusi_sumber.json tersimpan
  [HDFS] /data/news/hasil/volume_per_jam.json tersimpan
Semua hasil analisis tersimpan ke HDFS!


In [24]:
# Simpan juga ke lokal untuk Dashboard
results = {
    "kata_trending": df_kata.toPandas().to_dict(orient="records"),
    "distribusi_sumber": df_sumber.toPandas().to_dict(orient="records"),
    "volume_per_jam": df_jam.toPandas().to_dict(orient="records"),
    "generated_at": __import__("datetime").datetime.now().isoformat()
}
output_path = os.path.join("..", "dashboard", "data", "spark_results.json")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"spark_results.json tersimpan")
print(f"  {len(results['kata_trending'])} kata, {len(results['distribusi_sumber'])} sumber, {len(results['volume_per_jam'])} jam")

spark_results.json tersimpan
  15 kata, 15 sumber, 18 jam


---
## BONUS — Spark MLlib: K-Means Clustering (+5 poin)

In [25]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

indo_sw = ["dan","yang","di","ke","dari","untuk","dengan","ini","itu","ada",
  "akan","juga","tidak","se","pada","oleh","secara","sebuah","tersebut",
  "telah","saat","bisa","lebih","masih","sudah","hari","dalam","tahun","atau","ia","soal","usai"]

pipeline = Pipeline(stages=[
    Tokenizer(inputCol="judul_lower", outputCol="words"),
    StopWordsRemover(inputCol="words", outputCol="filtered", stopWords=indo_sw),
    HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=1000),
    IDF(inputCol="rawFeatures", outputCol="features"),
    KMeans(k=5, seed=42, featuresCol="features", predictionCol="cluster")
])

df_ml = df_all.withColumn("judul_lower", lower(col("judul"))).filter(col("judul").isNotNull())
model = pipeline.fit(df_ml)
predictions = model.transform(df_ml)
print("K-Means selesai! Distribusi cluster:")
predictions.groupBy("cluster").count().orderBy("cluster").show()

K-Means selesai! Distribusi cluster:
+-------+-----+
|cluster|count|
+-------+-----+
|      0|    1|
|      1|   77|
|      2|    2|
|      3|    1|
|      4|    1|
+-------+-----+



In [26]:
print("Contoh judul per cluster:\n")
for i in range(5):
    print(f"Cluster {i}")
    for row in predictions.filter(col("cluster")==i).select("judul","sumber").take(3):
        print(f"  [{row['sumber']}] {row['judul'][:80]}")
    print()

Contoh judul per cluster:

Cluster 0
  [Tempo] Pesan Mendikti di Hardiknas 2026: Kampus Harus Responsif

Cluster 1
  [Kompas.com] Soal Kelahiran Anak Al dan Alyssa, Ahmad Dhani: Mudah-mudahan Malam Ini
  [detikHOT] Pengakuan Aurel Baru Sebulan Nikah Minta Pisah dan Sikap Atta Halilintar
  [detikHOT] Ivan Gunawan Riset Lokasi Pembangunan 99 Masjid dari Sabang Sampai Merauke

Cluster 2
  [Tempo] Megawati: Buruh Bukan Sekadar Faktor Produksi
  [tandaseru.id] Lonjakan Permintaan Mac Mini untuk AI, Apple Hentikan Produksi Model Termurah

Cluster 3
  [Tempo] Peserta LPDP Soroti Larangan Pegang HP Selama Pembekalan di Kompleks TNI

Cluster 4
  [Tempo] Pegadaian Salurkan Bantuan untuk Masyarakat Medan Lewat Program Sedekah Jumat



In [27]:
print("Kata dominan per cluster:\n")
for i in range(5):
    kc = predictions.filter(col("cluster")==i).select("judul_lower") \
        .select(explode(split(col("judul_lower"),r"\s+")).alias("kata")) \
        .withColumn("kata",trim(regexp_replace(col("kata"),r"[^a-zA-Z0-9]",""))) \
        .filter(~col("kata").isin(STOPWORDS)).filter(length(col("kata"))>1) \
        .groupBy("kata").count().orderBy(desc("count")).take(5)
    print(f"  Cluster {i}: {', '.join(f'{r.kata}({r[1]})' for r in kc)}")

Kata dominan per cluster:

  Cluster 0: 2026(1), harus(1), pesan(1), mendikti(1), hardiknas(1)
  Cluster 1: 2026(10), jadi(7), prabowo(7), megawati(6), hardiknas(5)
  Cluster 2: produksi(2), permintaan(1), apple(1), bukan(1), model(1)
  Cluster 3: peserta(1), tni(1), pembekalan(1), soroti(1), kompleks(1)
  Cluster 4: bantuan(1), program(1), masyarakat(1), medan(1), sedekah(1)


### Interpretasi MLlib K-Means
1. **Identifikasi topik** — setiap cluster = satu tema besar
2. **Trend detection** — cluster terbesar = isu dominan
3. **Strategic planning** — petakan klien ke cluster relevan
4. **Early warning** — cluster baru = potensi breaking news

In [28]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
